In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

print(train.shape)
print(test.shape)

train.head()

In [ ]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]

X = train[features]
y = train["Survived"]

X_test = test[features]

print(X.head())
print(y.head())
print(X_test.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 元データを壊さないようにコピー
X = X.copy()
X_test = X_test.copy()

# 欠損値を補完
X["Age"] = X["Age"].fillna(X["Age"].median())
X_test["Age"] = X_test["Age"].fillna(X["Age"].median())

X["Embarked"] = X["Embarked"].fillna(X["Embarked"].mode()[0])
X_test["Embarked"] = X_test["Embarked"].fillna(X["Embarked"].mode()[0])

X_test["Fare"] = X_test["Fare"].fillna(X["Fare"].median())

# 文字列を数値に変換
X = pd.get_dummies(X, columns=["Sex", "Embarked"])
X_test = pd.get_dummies(X_test, columns=["Sex", "Embarked"])

# train側とtest側の列をそろえる
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print(X.head())
print(X_test.head())

In [ ]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def load_data():
    """
    Kaggle Notebook 上の Titanic データを読み込む。
    """
    train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
    test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
    return train, test


def extract_title(df):
    """
    Name から敬称を抽出する。
    例:
    Braund, Mr. Owen Harris -> Mr
    Cumings, Mrs. John Bradley -> Mrs
    """
    df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)

    # 少数派の敬称をまとめる
    title_mapping = {
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs"
    }
    df["Title"] = df["Title"].replace(title_mapping)

    rare_titles = [
        "Lady", "Countess", "Capt", "Col", "Don", "Dr",
        "Major", "Rev", "Sir", "Jonkheer", "Dona"
    ]
    df["Title"] = df["Title"].replace(rare_titles, "Rare")

    return df


def create_features(train, test):
    """
    train と test を結合して、同じ前処理を行う。
    その後、再度 train/test に分割する。
    """
    train = train.copy()
    test = test.copy()

    train["is_train"] = 1
    test["is_train"] = 0

    combined = pd.concat([train, test], axis=0, sort=False)

    # 追加特徴量1: Name から Title を抽出
    combined = extract_title(combined)

    # 追加特徴量2: 家族人数
    combined["FamilySize"] = combined["SibSp"] + combined["Parch"] + 1

    # 追加特徴量3: 一人乗船かどうか
    combined["IsAlone"] = (combined["FamilySize"] == 1).astype(int)

    # 追加特徴量4: Cabin 情報があるかどうか
    combined["HasCabin"] = combined["Cabin"].notnull().astype(int)

    # 欠損値補完
    combined["Age"] = combined["Age"].fillna(combined["Age"].median())
    combined["Fare"] = combined["Fare"].fillna(combined["Fare"].median())
    combined["Embarked"] = combined["Embarked"].fillna(combined["Embarked"].mode()[0])

    features = [
        "Pclass",
        "Sex",
        "Age",
        "SibSp",
        "Parch",
        "Fare",
        "Embarked",
        "Title",
        "FamilySize",
        "IsAlone",
        "HasCabin",
    ]

    X_all = combined[features]

    # カテゴリ変数を one-hot encoding
    X_all = pd.get_dummies(
        X_all,
        columns=["Sex", "Embarked", "Title"],
        drop_first=True
    )

    X_train = X_all[combined["is_train"] == 1]
    X_test = X_all[combined["is_train"] == 0]

    y = combined.loc[combined["is_train"] == 1, "Survived"].astype(int)

    return X_train, y, X_test


def compare_models(X, y):
    """
    複数モデルを Cross Validation で比較する。
    """
    models = {
        "LogisticRegression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, random_state=42))
        ]),
        "RandomForest": RandomForestClassifier(
            n_estimators=200,
            max_depth=5,
            random_state=42
        ),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        )
    }

    print("Cross Validation Results")
    print("-" * 40)

    best_model_name = None
    best_score = 0
    best_model = None

    for name, model in models.items():
        scores = cross_val_score(
            model,
            X,
            y,
            cv=5,
            scoring="accuracy"
        )

        mean_score = scores.mean()
        std_score = scores.std()

        print(f"{name}: {mean_score:.4f} (+/- {std_score:.4f})")

        if mean_score > best_score:
            best_score = mean_score
            best_model_name = name
            best_model = model

    print("-" * 40)
    print(f"Best Model: {best_model_name}")
    print(f"Best CV Score: {best_score:.4f}")

    return best_model_name, best_model


def create_submission(model, X, y, X_test, test):
    """
    全学習データで再学習し、提出ファイルを作成する。
    """
    model.fit(X, y)
    predictions = model.predict(X_test)

    submission = pd.DataFrame({
        "PassengerId": test["PassengerId"],
        "Survived": predictions.astype(int)
    })

    submission.to_csv("submission_v2.csv", index=False)
    print("submission_v2.csv を作成しました。")
    print(submission.head())


def main():
    train, test = load_data()

    print("train shape:", train.shape)
    print("test shape:", test.shape)

    X, y, X_test = create_features(train, test)

    print("X shape:", X.shape)
    print("X_test shape:", X_test.shape)
    print("features:", list(X.columns))

    best_model_name, best_model = compare_models(X, y)

    create_submission(best_model, X, y, X_test, test)


if __name__ == "__main__":
    main()

In [ ]:
import os

os.listdir("/kaggle/working")